# FeatureLens offline study

This notebook runs the full FeatureLens empirical study on a CUDA runtime while persisting experiment artifacts to Google Drive. It is designed to be resumable after Colab disconnects.

**Before running:** choose a GPU runtime in Colab, then execute the cells from top to bottom.

In [ ]:
# 1. Verify that Colab actually assigned a GPU.
import subprocess, sys

subprocess.run(["nvidia-smi"], check=True)

try:
    import torch
    assert torch.cuda.is_available(), "CUDA is not available. Change the Colab runtime to a GPU and reconnect."
    props = torch.cuda.get_device_properties(0)
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram_gb = props.total_memory / 1024**3
    print(f"\nGPU: {gpu_name} | VRAM: {gpu_vram_gb:.1f} GB")
except Exception as exc:
    raise RuntimeError("A CUDA GPU runtime is required for the model stages.") from exc

In [ ]:
# 2. Mount Google Drive so completed experiment stages survive a runtime reset.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. Configuration — edit REPO_URL before running this cell.
from pathlib import Path

REPO_URL = "PASTE_YOUR_GIT_REPO_URL_HERE"
BRANCH = "main"
DRIVE_RUN_NAME = "FeatureLens_offline_v015"

REPO_DIR = Path("/content/FeatureLens")
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_RUN_NAME
DRIVE_ARTIFACTS = DRIVE_ROOT / "artifacts"
LOG_PATH = DRIVE_ROOT / "offline_study.log"

if REPO_URL.startswith("PASTE_"):
    raise ValueError("Set REPO_URL to your FeatureLens Git repository URL first.")

DRIVE_ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("Persistent run directory:", DRIVE_ROOT)

In [ ]:
# 4. Clone or refresh the FeatureLens source.
import shutil, subprocess

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

print(subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
# 5. Install the project environment. This can take a few minutes on a fresh runtime.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)

In [ ]:
# 6. Re-check CUDA after dependency installation and choose a conservative activation batch.
import os, torch

assert torch.cuda.is_available(), "CUDA disappeared after dependency setup."
gpu_name = torch.cuda.get_device_name(0)
gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
ACTIVATION_BATCH_SIZE = 16 if gpu_vram_gb >= 20 else 8
ACTIVATION_MAX_LENGTH = 192

# Keep model/SAE downloads on Colab's local disk for speed.
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print(f"GPU: {gpu_name} ({gpu_vram_gb:.1f} GB)")
print(f"Activation batch size: {ACTIVATION_BATCH_SIZE}")

In [ ]:
# 7. Link FeatureLens artifacts to Google Drive.
# Existing small repo artifacts (for example README.md) are copied once; the local directory is then replaced by a symlink.
import shutil

local_artifacts = REPO_DIR / "artifacts"
if local_artifacts.is_symlink():
    local_artifacts.unlink()
elif local_artifacts.exists():
    shutil.copytree(local_artifacts, DRIVE_ARTIFACTS, dirs_exist_ok=True)
    shutil.rmtree(local_artifacts)

local_artifacts.symlink_to(DRIVE_ARTIFACTS, target_is_directory=True)
print("artifacts ->", local_artifacts.resolve())

## Run / resume the study

The command below is safe to rerun. Completed stages are skipped. The causal and feature-set stages also checkpoint completed tasks, so a disconnect during either stage does not discard earlier tasks from that stage.

In [ ]:
# 8. Run the full pipeline with live output and a persistent log.
import subprocess, sys, time

command = [
    sys.executable, "-m", "experiments.run_all",
    "--resume",
    "--activation-batch-size", str(ACTIVATION_BATCH_SIZE),
    "--activation-max-length", str(ACTIVATION_MAX_LENGTH),
]

print("$", " ".join(command))
print("Log:", LOG_PATH)
start = time.time()

with LOG_PATH.open("a", encoding="utf-8") as log:
    log.write("\n\n=== FeatureLens run ===\n")
    log.write("$ " + " ".join(command) + "\n")
    process = subprocess.Popen(
        command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Pipeline exited with code {return_code}. Fix the error, then rerun this cell; --resume will keep completed work."
    )

print(f"\nCompleted in {(time.time() - start) / 60:.1f} minutes.")

In [ ]:
# 9. Validate the measured artifact set.
import subprocess, sys
subprocess.run([sys.executable, "-m", "scripts.validate_artifacts"], cwd=REPO_DIR, check=True)

In [ ]:
# 10. Inspect the study summary and report.
from pathlib import Path
import json, pandas as pd
from IPython.display import display, Markdown

summary_path = DRIVE_ARTIFACTS / "study_summary.json"
study_table_path = DRIVE_ARTIFACTS / "study_feature_summary.csv"
report_path = DRIVE_ARTIFACTS / "report.md"

summary = json.loads(summary_path.read_text(encoding="utf-8"))
display(summary)
display(pd.read_csv(study_table_path))
display(Markdown(report_path.read_text(encoding="utf-8")))

In [ ]:
# 11. Create a small publishable artifact bundle (activation caches and checkpoint markers are excluded).
import zipfile

PUBLISH_ZIP = DRIVE_ROOT / "FeatureLens_offline_results.zip"

with zipfile.ZipFile(PUBLISH_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(DRIVE_ARTIFACTS.rglob("*")):
        if not path.is_file():
            continue
        rel = path.relative_to(DRIVE_ARTIFACTS)
        if rel.parts and rel.parts[0] == "activations":
            continue
        if path.name.endswith(".complete") or path.name.endswith(".tmp"):
            continue
        zf.write(path, arcname=str(Path("artifacts") / rel))

print("Publishable bundle:", PUBLISH_ZIP)
print(f"Size: {PUBLISH_ZIP.stat().st_size / 1024**2:.2f} MiB")

## After Colab

Download `FeatureLens_offline_results.zip` from the Drive run folder. Extract it over your local FeatureLens repository so the files land under `artifacts/`, run the normal release checks locally, inspect the measured report, and only then commit the small study artifacts. Do **not** commit `artifacts/activations/`.